In [39]:
import sys
PATH_TO_PREDICTOR_HOME = ".."
sys.path.append(PATH_TO_PREDICTOR_HOME)


from mhcnames import normalize_allele_name
import numpy as np
import pandas as pd
from config import RESULT_COLUMNS


# 0. Read the data

In [41]:
from config import PER_ORGANISM_PER_CLASS_ALLELES
from mhcgnomes import parse

DATA_PATH = "/home/denklewer/Data/schreiber_full/"
# read total data

total_df =  pd.read_csv("{}schreiber_total_data_full.csv".format(DATA_PATH),sep=";")
ALLELES = list(set(total_df['allele']))
per_allele_mt_df =dict()
per_allele_wt_df = dict()
for allele_name in ALLELES:
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    df_mt = pd.read_csv("{}schreiber_data_full_{}_mt.csv".format(DATA_PATH,target_allele_name),sep=";")
    df_wt = pd.read_csv("{}schreiber_data_full_{}_wt.csv".format(DATA_PATH,target_allele_name),sep=";") 
    
    per_allele_mt_df[allele_name] = df_mt
    per_allele_wt_df[allele_name] = df_wt
    
    assert 'peptide' in df_mt.columns
    assert 'length' in df_mt.columns
    assert RESULT_COLUMNS['id'] in df_mt.columns
    assert 'peptide' in df_wt.columns
    assert 'length' in df_wt.columns
    assert RESULT_COLUMNS['id'] in df_wt.columns
    
    
def defineOrganism(allele_name):
    parse_result = parse(allele_name)
    if parse_result.is_mouse:
        return "Mouse"
    elif parse_result.is_human:
        return "Human"
    else:
        print("organism not found")
        return None


def defineClass(allele_name):
    parse_result = parse(allele_name)
    if parse_result.has_mhc_class:
        if parse_result.is_class1:
            return 'I'
        elif parse_result.is_class2:
            return 'II'
        else:
            print("Not class I or class II???!")
    else:
        print("Result without class")
    return None
ALLELES

['H-2-IAd', 'H-2-IEd', 'H-2-Dd', 'H-2-Ld', 'H-2-Kd']

In [42]:
per_method_per_allele_wt_predictions = dict()
per_method_per_allele_mt_predictions = dict()

In [43]:
METHOD = "mhcnuggets"

## 0.1. make result dicts

In [44]:
per_method_per_allele_wt_predictions = dict()
per_method_per_allele_mt_predictions = dict()

## 0.2 Read decoy data

In [45]:


per_allele_decoy = dict()
for allele_name in ALLELES:
    organism =  defineOrganism(allele_name).lower()
    class_to_use = defineClass(allele_name)
    DECOY_DATA_PATH = f"/home/denklewer/Data/decoy/sampled_decoy_data_{organism}_class_{class_to_use}.csv"
    decoy_df =  pd.read_csv(DECOY_DATA_PATH)
    decoy_df['length'] = decoy_df['peptide'].str.len()
    
    per_allele_decoy[allele_name] = decoy_df.copy()

# 1. Predict with mhcnuggets

In [46]:
# importing the predict module
from mhcnuggets.src.predict import predict
from mhcnames import normalize_allele_name## ONLY FOR LINUX
import numpy as np
import pandas as pd

Using TensorFlow backend.


In [47]:
            
def filterByLength(per_allele_df, length_column='length'):
    #Filter for mhcnuggets <=15 length for class 1 and <=30 for class 2
    for allele_name in per_allele_df.keys():
        
        initial_len = len(per_allele_df[allele_name])
        if defineClass(allele_name) == 'I':
            per_allele_df[allele_name] = per_allele_df[allele_name][per_allele_df[allele_name][length_column] <= 15]
        else:
            per_allele_df[allele_name] = per_allele_df[allele_name][per_allele_df[allele_name][length_column] <= 30]
        result_len = len(per_allele_df[allele_name])
        print(f"Filtered out {initial_len - result_len} samples")
        

    
    

In [48]:
%%time
print("decoy ")
filterByLength(per_allele_decoy)

print(f"\t mt")
filterByLength(per_allele_mt_df)
print(f"\t wt")
filterByLength(per_allele_wt_df)

decoy 
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
	 mt
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
	 wt
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
CPU times: user 102 ms, sys: 8.58 ms, total: 111 ms
Wall time: 111 ms


In [49]:
# different allele naming.
TARGET_ALLELES = dict()

## HARDCODE
for allele_name in ALLELES:
    if allele_name == 'H2_d class II':
        target_allele_name = 'H2_IAd'
    elif allele_name == 'H2_b class II':
        target_allele_name = 'H2_IAb'
    elif allele_name == 'HLA_DRB1_01_01':
        target_allele_name = 'HLA-DRB101:01'
    elif allele_name == 'HLA_DRB1_04_01':
        target_allele_name = 'HLA-DRB104:01'
    elif allele_name == 'HLA_DRB1_04_05':
        target_allele_name = 'HLA-DRB104:05'    
    else:
        target_allele_name = normalize_allele_name(allele_name).replace("*", "")
    TARGET_ALLELES[allele_name] = (target_allele_name)


TARGET_ALLELES

{'H-2-IAd': 'H-2-IAd',
 'H-2-IEd': 'H-2-IEd',
 'H-2-Dd': 'H-2-Dd',
 'H-2-Ld': 'H-2-Ld',
 'H-2-Kd': 'H-2-Kd'}

In [121]:
# predicting new line separated peptides present in the peptides_path file 
# for MHC class_I allele HLA-A02:01. Since a MS_BA blend model is available,
# this prediction will use the blended model (HLA-A02:01_BA_to_HLAp.h5)
def predict_with_mhcnuggets(per_allele_df, per_allele_scores_for_pecentiles = None):
    # save to temp dir
    TEMPDIR = '../TMP/'
    for allele_name in ALLELES:
        df =  per_allele_df[allele_name].copy()
        initial_len = len(df)
        df =  df[df['peptide'].notnull()]['peptide']
        null_deleted = initial_len - len(df)
        df = df.drop_duplicates() # remove nans
        dup_deleted = initial_len - null_deleted - len(df)
        print(f" initial len {initial_len} -> {len(df)}(null:{null_deleted}, dup:{dup_deleted})")
        df.to_csv(f"{TEMPDIR}mchnuggets_data_{allele_name}", index=False, header=False)    
    # predict
    for allele_name in ALLELES:
        class_to_use = defineClass(allele_name)
        target_allele = TARGET_ALLELES[allele_name]
        print(f"---{target_allele}")
        predict(class_=f'{class_to_use}',
                peptides_path=f'{TEMPDIR}mchnuggets_data_{allele_name}', 
                mhc=target_allele,
                output=f'{TEMPDIR}mchnuggets_predictions_{allele_name}')
    # read predictions from temp
    per_allele_predictions= {
        allele_name :pd.read_csv(f"{TEMPDIR}mchnuggets_predictions_{allele_name}").rename(columns={'ic50':'affinity'}) for allele_name in ALLELES
    }
        
    for allele_name in per_allele_predictions.keys(): 
        df_predictions = per_allele_predictions[allele_name]
        
        df_source = per_allele_df[allele_name].copy()
        assert len(df_source[df_source['peptide'].notnull()]['peptide'].unique()) == len(per_allele_predictions[allele_name])
        
        
        df_result = df_source.merge(df_predictions, on=['peptide'], how='left')

    
        # log-scale affinity
        
        df_result['score'] = 1 - np.log(df_result['affinity'])/np.log(50000)
        
        if per_allele_scores_for_pecentiles:
            
            # get scores for percentile
            scores = df_result['score']
            ref_scores = per_allele_scores_for_pecentiles[allele_name].copy()
            ref_scores = ref_scores['score'].sort_values()
            pct_increment = 1./len(ref_scores)
            df_result['percentile_rank'] = (0.5 + 0.5*ref_scores.searchsorted(scores, side='left') + 0.5*ref_scores.searchsorted(scores, side='right'))*pct_increment
            df_result['percentile_rank'] = 100 - df_result['percentile_rank']*100
            
            # add binder label
        per_allele_predictions[allele_name] = df_result
            
    
    
    return per_allele_predictions
    

## create tempfiles for mhcnuggets



In [118]:
df = per_allele_wt_df[ALLELES[2]]
len(df), len(df['peptide'].unique())

(8738, 8700)

In [119]:
len(per_allele_wt_predictions[ALLELES[3]])

8862

## First calculate scores for decoy

In [120]:
%%time
per_allele_decoy_predictions = predict_with_mhcnuggets(per_allele_decoy)

 initial len 200000 -> 200000(null:0, dup:0)
 initial len 200000 -> 200000(null:0, dup:0)
 initial len 200000 -> 200000(null:0, dup:0)
 initial len 200000 -> 200000(null:0, dup:0)
 initial len 200000 -> 200000(null:0, dup:0)
---H-2-IAd
Predicting for 200000 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found H-2-IAd
No BA_to_HLAp model found, predicting with BA model...
---H-2-IEd
Predicting for 200000 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found H-2-IEd
No BA_to_HLAp model found, predicting with BA model...
---H-2-Dd
Predicting for 200000 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found H-2-Dd
No BA_to_HLAp model found, predicting with BA model...
---H-2-Ld
Predicting for 200000 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found H-2-Ld
No BA_to_HLAp model found, predicting with BA model...
---

In [122]:
%%time
print(f"--mt")
per_allele_mt_predictions = predict_with_mhcnuggets(per_allele_mt_df, per_allele_decoy_predictions)
per_method_per_allele_mt_predictions[METHOD] = per_allele_mt_predictions  

print(f"--wt")
per_allele_wt_predictions = predict_with_mhcnuggets(per_allele_wt_df, per_allele_decoy_predictions)
per_method_per_allele_wt_predictions[METHOD] = per_allele_wt_predictions  


--mt
 initial len 52091 -> 52091(null:0, dup:0)
 initial len 52091 -> 52091(null:0, dup:0)
 initial len 8738 -> 8738(null:0, dup:0)
 initial len 8738 -> 8738(null:0, dup:0)
 initial len 8738 -> 8738(null:0, dup:0)
---H-2-IAd
Predicting for 52091 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found H-2-IAd
No BA_to_HLAp model found, predicting with BA model...
---H-2-IEd
Predicting for 52091 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found H-2-IEd
No BA_to_HLAp model found, predicting with BA model...
---H-2-Dd
Predicting for 8738 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found H-2-Dd
No BA_to_HLAp model found, predicting with BA model...
---H-2-Ld
Predicting for 8738 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found H-2-Ld
No BA_to_HLAp model found, predicting with BA model...
---H-2-Kd
Predicting

In [123]:
per_method_per_allele_wt_predictions[METHOD][ALLELES[2]]

,source_sequence_name,allele,peptide,length,affinity,score,percentile_rank
0,seq124,H-2-Dd,SALSILLL,8,19525.50,0.086906,14.58275
1,seq194,H-2-Dd,SVTAVAETQ,9,36502.88,0.029079,80.19375
2,seq329,H-2-Dd,SVTAVAET,8,36825.66,0.028266,82.49375
3,seq401,H-2-Dd,GGDCLLLL,8,6635.94,0.186651,3.10675
4,seq431,H-2-Dd,DVINSRTQ,8,33149.65,0.037985,57.29725
...,...,...,...,...,...,...,...
8733,seq26209,H-2-Dd,MGVSSLQGIQF,11,6332.06,0.190983,2.94475
8734,seq26210,H-2-Dd,GVSSLQGIQFF,11,2189.52,0.289132,1.04625
8735,seq26211,H-2-Dd,VSSLQGIQFFD,11,25846.43,0.060986,26.77925
8736,seq26212,H-2-Dd,VSSLQGIQFF,10,28159.76,0.053063,33.82775


# 2. Join predictions

In [124]:
per_method_columns = {
    'netMHC-3.4': ['source_sequence_name' ,'score', 'affinity', 'percentile_rank'],
    'netMHC-4.0':['source_sequence_name', 'score', 'affinity', 'percentile_rank' ],
    'netMHCpan-4.1':['source_sequence_name', 'score_ba', 'score_el', 'affinity', 'percentile_rank_ba', 'percentile_rank_el'],
    'netMHCIIpan-4.0':['source_sequence_name', 'score_ba', 'score_el', 'affinity', 'percentile_rank_ba', 'percentile_rank_el'],
    'mhcnuggets':['source_sequence_name', 'score', 'affinity', 'percentile_rank']

}



def rename_columns(df, method):
    leave_columns = ['source_sequence_name', 'peptide']
    old_columns = [col for col in per_method_columns[method] if col not in leave_columns]
    new_columns = [method + "_" + col for col in old_columns]
    df = df.rename(columns={old:new for old,new in zip(old_columns,new_columns)})
    return df


def getPredictionResult(total_results, allele_name, method):
    df = total_results[method][allele_name][per_method_columns[method]].copy()
    df = rename_columns(df, method)
    return df
    

# def getPredictionResult(total_results, region, allele_name, method):
#     df = total_results[region][method][allele_name][per_method_columns[method]].copy()
#     df = rename_columns(df, method)
#     return df


In [125]:
def joinPredictionsToSource(per_method_per_allele_mt_predictions, per_method_per_allele_wt_predictions):
    per_allele_result = dict()
    methods = list(per_method_per_allele_mt_predictions.keys())
    for allele_name in ALLELES:
        print(f"allele {allele_name}")
        df_source_wt = per_allele_wt_df[allele_name].copy()
        df_source_mt = per_allele_mt_df[allele_name].copy()

        for method in methods:
            print(f"method {method}")
            df_method_wt = getPredictionResult(per_method_per_allele_wt_predictions, allele_name, method)
            df_method_mt = getPredictionResult(per_method_per_allele_mt_predictions, allele_name, method)
            df_source_wt = df_source_wt.merge(df_method_wt, on=['source_sequence_name'], how='left')
            df_source_mt = df_source_mt.merge(df_method_mt, on=['source_sequence_name'], how='left')
        leave_columns = [ 'source_sequence_name', 'allele', 'length']
        df_source_wt = df_source_wt.rename(columns={col:  col + "_wt" for col in df_source_wt.columns if col not in leave_columns})
        df_source_mt = df_source_mt.rename(columns={col:  col + "_mt" for col in df_source_mt.columns if col not in leave_columns})
        # rename id back 


        df_result = df_source_mt.merge(df_source_wt, on=['source_sequence_name', 'allele', 'length'], how='left')
        per_allele_result[allele_name] = df_result
    return per_allele_result
                
                

              

In [126]:
per_allele_result = joinPredictionsToSource(per_method_per_allele_mt_predictions, per_method_per_allele_wt_predictions)

allele H-2-IAd
method mhcnuggets
allele H-2-IEd
method mhcnuggets
allele H-2-Dd
method mhcnuggets
allele H-2-Ld
method mhcnuggets
allele H-2-Kd
method mhcnuggets


# 5. For now just write to files

In [127]:
TARGET_PATH_TO_PREDICTIONS = DATA_PATH + f'predictions/{METHOD}/'

import os
if not os.path.exists(TARGET_PATH_TO_PREDICTIONS):
    os.makedirs(TARGET_PATH_TO_PREDICTIONS)

In [128]:
# for decoy_key in DECOY_PREFIXES:

for allele_name in ALLELES:
    total_df = per_allele_result[allele_name]
    allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    total_df.to_csv("{}{}_predictions_allele_{}.csv".format(TARGET_PATH_TO_PREDICTIONS, METHOD,allele_name),sep=";", index=False)

In [129]:
total_df

,source_sequence_name,allele,peptide_mt,length,mhcnuggets_score_mt,mhcnuggets_affinity_mt,mhcnuggets_percentile_rank_mt,peptide_wt,mhcnuggets_score_wt,mhcnuggets_affinity_wt,mhcnuggets_percentile_rank_wt
0,seq0,H-2-Kd,KYLTQRENL,9,0.565869,109.64,0.14075,KYLTQREKL,0.493631,239.56,0.32225
1,seq6,H-2-Kd,VYQVERERL,9,0.365231,961.08,1.27325,VYQVELERL,0.378903,828.93,1.11025
2,seq11,H-2-Kd,RFAPQAELL,9,0.225169,4374.31,5.21925,RFPPQAELL,0.183797,6844.09,8.09625
3,seq14,H-2-Kd,APSNSGAGL,9,0.031648,35502.37,72.95675,APSNSGADL,0.031345,35619.05,73.39875
4,seq15,H-2-Kd,SFVAVVITI,9,0.632756,53.17,0.06125,SFVAAVITI,0.625665,57.41,0.06875
...,...,...,...,...,...,...,...,...,...,...,...
8733,seq14195,H-2-Kd,PTTVCQPN,8,0.037183,33438.61,65.06225,ATTVCQPN,0.036197,33797.39,66.39875
8734,seq14197,H-2-Kd,PRLHHSSR,8,0.048203,29680.02,51.92925,PLLHHSSR,0.080422,20944.43,29.48525
8735,seq14200,H-2-Kd,PPRLHHSSRSC,11,0.050421,28976.42,49.68175,PPLLHHSSRSC,0.057496,26841.04,43.50575
8736,seq14201,H-2-Kd,PNNISAIS,8,0.047889,29781.27,52.24375,PNNISGIS,0.034705,34347.13,68.48625
